# MONAI Label and 3D Slicer (SlicerJupyter) DICOM Example - Whole Brain Segmentation
<hr/>
***As-Is Software Disclaimer***

The content in this repository is delivered “As-Is”. Notwithstanding anything to the contrary, DNAnexus will have no warranty, support, liability or other obligations with respect to Materials provided hereunder.

<hr/>

This notebook provides a demonstration of auto-segmentation and model training on DICOM imaging data, utilizing MONAI Label and 3D Slicer (SlicerJupyter). 

Note: Data preprocessing may be required for optimal model performance, and the segmentation results shown here are for illustration and not guaranteed to be accurate.

For general examples of using Slicer within Jupyter notebook, see: [Slicer/SlicerNotebooks](https://github.com/Slicer/SlicerNotebooks).

For MONAI Label example notebooks and tutorials, see [Project-MONAI/tutorials/monailabel](https://github.com/Project-MONAI/tutorials/tree/main/monailabel).


<a href="https://github.com/dnanexus/OpenBio/blob/master/LICENSE.md">MIT License</a> applies to this notebook.

## Preparing your environment
### Launch spec:

* App name: JupyterLab with Python, R, Stata, ML, Image Processing
* App flavor: MONAI_ML
* Kernel: Slicer 5.8
* Instance type: mem2_ssd1_v2_x16
* Runtime: =~ 20 min + 10 min initialization (one sample)
* Data description: Input for this notebook is  Brain MRI, T1 structural dataset in DICOM format. The dataset can be downloaded from https://biobank.ndph.ox.ac.uk/showcase/ukb/examples/eg_brain_t1.zip.
  * Target: Brain
  * Modality: MRI

### Package and tools dependency:

| Package | License | 
| --- | --- |
| <a href="https://github.com/rordenlab/dcm2niix">dcm2niix</a> | <a href="https://github.com/rordenlab/dcm2niix?tab=License-1-ov-file">BSD 3-Clause License</a> |

### Imports

In [ ]:
import os

import JupyterNotebooksLib as slicernb
import slicer
import vtk
from DICOMLib import DICOMUtils
from ipywidgets import Button, HBox, interact
from MONAILabel import MONAILabelLogic

## 1. Prepare DICOM Data

In [ ]:
DICOM_DIR = os.path.join(os.getcwd(), "eg_brain_t1")
! export DICOM_DIR="${pwd}/eg_brain_t1"

### Download data

In [ ]:
! wget -nd biobank.ndph.ox.ac.uk/ukb/ukb/examples/eg_brain_t1.zip

In [ ]:
! apt-get -qq update && apt-get install unzip
! unzip -qq eg_brain_t1.zip -d $DICOM_DIR

### Option 1: Convert to NIfTI
This is often the simplest approach for local DICOM directories. For example, use a tool like `dcm2niix` to convert your DICOM series into single NIfTI volumes, which MONAI Label can easily load. Start the server pointing to directory containing converted NIfTI files.

In [ ]:
! apt-get update && apt-get -y install dcm2niix

In [ ]:
! mkdir -p eg_brain_t1_nifti_1
! dcm2niix -o eg_brain_t1_nifti_1 $DICOM_DIR

### Option 2: Use the DICOM Module in 3D Slicer
Load your DICOM directory directly into the Slicer DICOM database and then either export entire DICOM database to research file format or use the MONAI Label module to upload the loaded volume to the server.

#### Load DICOM files into Slicer DICOM database

In [ ]:
# Switch to DICOM module
slicer.util.selectModule("DICOM")

# Instantiate a new DICOM browser
dicomBrowser = slicer.modules.DICOMWidget.browserWidget.dicomBrowser

# Create database directory
dicomBrowser.createNewDatabaseDirectory()

# Import directory containing DICOM files
dicomBrowser.importDirectory(DICOM_DIR)

# Wait for import to finish before proceeding (optional, if removed then import runs in the background)
dicomBrowser.waitForImportFinished()

print(f"Database stored at: {slicer.dicomDatabase.databaseDirectory}")

#### Explore loaded data in the Slicer DICOM database

In [ ]:
db = slicer.dicomDatabase

patientList = list(db.patients())
print(f"Patients: {patientList}")

studyList = list(db.studiesForPatient(patientList[0]))
print(f"Studies for patient {patientList[0]}: {studyList}")

seriesList = list(db.seriesForStudy(studyList[0]))
print(f"Series for study {studyList[0]}: {seriesList}")

fileList = db.filesForSeries(seriesList[0])
print(f"Number of files for series {seriesList[0]}: {len(fileList)}")

#### 2.a Export DICOM series from the database to research file format
Export the entire Slicer DICOM database content to nifti (or nrrd, etc.) file format with filtering of data type and naming of the output file based on DICOM tags. Start the server pointing to this directory.

In [ ]:
outputFolder = os.path.join(os.getcwd(), "eg_brain_t1_nifti_2/")

patientUIDs = slicer.dicomDatabase.patients()
for patientUID in patientUIDs:
    loadedNodeIDs = DICOMUtils.loadPatientByUID(patientUID)
    for loadedNodeID in loadedNodeIDs:
        # Check if we want to save this node
        node = slicer.mrmlScene.GetNodeByID(loadedNodeID)
        # Only export images
        if not node or not node.IsA("vtkMRMLScalarVolumeNode"):
            continue
        # Construct filename
        shNode = slicer.mrmlScene.GetSubjectHierarchyNode()
        seriesItem = shNode.GetItemByDataNode(node)
        studyItem = shNode.GetItemParent(seriesItem)
        patientItem = shNode.GetItemParent(studyItem)
        filename = shNode.GetItemAttribute(patientItem, "DICOM.PatientID")
        filename += "_" + shNode.GetItemAttribute(studyItem, "DICOM.StudyDate")
        filename += "_" + shNode.GetItemAttribute(seriesItem, "DICOM.SeriesNumber")
        filename += "_" + shNode.GetItemAttribute(seriesItem, "DICOM.Modality")
        filename = (
            slicer.app.ioManager().forceFileNameValidCharacters(filename) + ".nii"
        )
        # Save node
        print(f"Write {node.GetName()} to {filename}")
        success = slicer.util.saveNode(node, outputFolder + filename)
    slicer.mrmlScene.Clear()

#### 2.b Use the MONAI Label module to upload the loaded volume to the server
Start MONAI Label server with --studies pointing to an empty directory, where the volumes will be later uploaded.

##### Load specific series to the Slicer scene

In [ ]:
seriesToLoad = seriesList[0]
print(f"Loading series {seriesToLoad} into the Slicer scene")

loadedNodeIDs = DICOMUtils.loadSeriesByUID([seriesList[0]])
print(f"Loaded node IDs: {loadedNodeIDs}")

##### Export loaded volume

In [ ]:
node = slicer.mrmlScene.GetNodeByID(loadedNodeIDs[0])

image_id = "image_1"
image_path = os.path.join(os.getcwd(), "image_1.nrrd")

slicer.util.saveNode(node, image_path)
print(f"Volume saved locally to: {image_path}")

After you start MONAI Label server with empty directory and connect from Slicer notebook, you can upload exported volume to the server:

```python
ml.upload_image(
    image_in=image_path,
    image_id=image_id
)
```


## 2. Prepare MONAI Label server
Run following `monailabel` commands in Terminal or in a notebook with Python 3 kernel.

### Download MONAI Bundle App

```bash
    monailabel apps --download --name monaibundle --output apps
```

### Start the MONAI Label server (with MONAI Bundle app)

Start the server with MONAI Bundle App pointing to directory with sample data. The chosen `wholeBrainSeg_Large_UNEST_segmentation` model will be automatically downloaded upon starting the MONAI Label server. For the --studies flag, the command expects a path and will not execute correctly if we do not supply one. Even if you plan to upload volumes from Slicer, a directory is required. 

```bash
    monailabel start_server --app apps/monaibundle --studies eg_brain_t1_nifti_2 --conf models wholeBrainSeg_Large_UNEST_segmentation 
```

## 3. Auto Segmentation with 3D Slicer (through SlicerJupyter)

### Connect to MONAI Label server

In [ ]:
SERVER_URL = "http://0.0.0.0:8000"

In [ ]:
ml = MONAILabelLogic()
ml.setServer(SERVER_URL)
ml.server_url

### Get server/app information

In [ ]:
ml.info()

### ( For option 2.b: Upload volume to the server - server started with empty datastore )

In [ ]:
ml.upload_image(image_in=image_path, image_id=image_id)

### Query datastore (cases/images on the server)

In [ ]:
ml.datastore()

### Load sample

Load image data from study datastore.

In [ ]:
sample = ml.next_sample(strategy="random")
image_id = sample["id"]
image_path = sample["path"]

print(sample)

### Run Auto Segmentation
Do the inference on currently loaded data. The pre-trained segmentation model is downloaded upon starting MONAI Label server in the `model` folder, in this case the model is saved at `apps/monaibundle/model/wholeBrainSeg_Large_UNEST_segmentation/models/model.pt`.

In [ ]:
label_file, params = ml.infer(
    model="wholeBrainSeg_Large_UNEST_segmentation", image_in=image_id
)

### Save/submit label

Save the ground truth label to the file system.

The final annotations will be saved to `labels/final` folder in the study dataset, for example, in this use case, the ground truth label will be saved to `eg_brain_t1_nifti_2/labels/final`.

In [ ]:
ml.save_label(image_id, label_file, params=params)

## 4. Visualize and edit annotations with 3D Slicer

### Set-up 3D Slicer scene

#### Configure 3D Slicer views

In [ ]:
# Set image viewer size to 70%
slicernb.AppWindow.setWindowSize(scale=0.7)
# Hide patient information from slice view
slicernb.showSliceViewAnnotations(False)

#### Clear scene

In [ ]:
slicer.mrmlScene.Clear(False)

#### Load image and label into 3D Slicer

In [ ]:
# Load image and label
image_node = slicer.util.loadVolume(image_path)
label_node = slicer.util.loadLabelVolume(label_file)

# Create segmentation node
segmentation_node = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSegmentationNode")
slicer.modules.segmentations.logic().ImportLabelmapToSegmentationNode(
    label_node, segmentation_node
)
# Set the segmentation geometry to match the image
segmentation_node.SetReferenceImageGeometryParameterFromVolumeNode(image_node)
# Generate a 3D surface model for visualization
segmentation_node.CreateClosedSurfaceRepresentation()

# Set the label overlay on the slice viewers
slicer.util.setSliceViewerLayers(background=image_node, label=segmentation_node)

### Visualize annotations

#### Static views

In [ ]:
slicernb.ViewDisplay(
    "FourUp"
)  # other views: OneUpRedSlice, OneUpGreenSlice, OneUpYellowSlice, OneUp3D

In [ ]:
# Lightbox view
slicernb.ViewLightboxDisplay("Red", rangeShrink=[45, 50])

#### (Optional) Update segment visibility (set to a specific segment)

In [ ]:
segmentation = segmentation_node.GetSegmentation()
displayNode = segmentation_node.GetDisplayNode()

# Define the target segment
targetSegmentName = "14"  # Right-Cerebral-White-Matter, see apps/monaibundle/model/wholeBrainSeg_Large_UNEST_segmentation/docs/README.md
targetSegmentID = segmentation.GetSegmentIdBySegmentName(targetSegmentName)

# Set segment visibility (making ONLY the target segment visible)
displayNode.SetAllSegmentsVisibility(False)
displayNode.SetSegmentVisibility(targetSegmentID, True)

##### Reset visibility to all segments

In [ ]:
segmentation = segmentation_node.GetSegmentation()
displayNode = segmentation_node.GetDisplayNode()

displayNode.SetAllSegmentsVisibility(True)

#### Dynamic views - Level 1. View objects + standard widgets
- Displayed content is saved in the notebook
- Views cannot be placed in a layout
- Low update rate (only for small adjustment of view parameters)
- Mouse and keyboard events are not captured

In [ ]:
# Slice view display
@interact(position=(0, 100))
def update(position=50):
    return slicernb.ViewSliceDisplay("Red", positionPercent=position)

#### Dynamic views - Level 2. View widgets
- Widgets can be placed in a layout
- Widget state (displayed content) is not saved in the notebook by default
- Low update rate (only for small adjustment of view parameters)
- Mouse and keyboard events are not captured

In [ ]:
# Slice widgets
slicernb.ViewSliceWidget("Red")
display(
    HBox(
        [
            slicernb.ViewSliceWidget("Red"),
            slicernb.ViewSliceWidget("Yellow"),
            slicernb.ViewSliceWidget("Green"),
        ]
    )
)

In [ ]:
# 3D widget
display(
    HBox(
        [
            slicernb.ViewSliceBaseWidget("Red", width="40%"),
            slicernb.View3DWidget(0, width="40%"),
        ]
    )
)

#### Dynamic views - Level 3. Interactive view widgets
- Some view controlling mouse and keyboard events are captured
- Only selected view can be displayed and controlled
- Medium update rate (somewhat usable on remote computers)

**Note:** Run cells with interactive widgets individually. Batch execution ("Run All Cells") is discouraged, as the cells may run indefinitely and cause execution interference.

In [ ]:
# Adjust maximum rate of Slicer's Jupyter kernel consuming Jupyter messages.
# Lower values make the notebook more responsive but too low values may make the Slicer application
# slow to respond.
slicer.modules.jupyterkernel.setPollIntervalSec(0.001)

# Add image 3D display
# slicernb.showVolumeRendering(image_node)

# 3D view
slicernb.AppWindow.setWindowSize(scale=0.8)
live3d = slicernb.ViewInteractiveWidget("1")
live3d.trackMouseMove = False
display(live3d)

In [ ]:
# Slice view (use arrow keys to move between slices, right-click-and-drag to zoom in/out)
liveRedSlice = slicernb.ViewInteractiveWidget("R")
liveRedSlice.trackMouseMove = False
display(liveRedSlice)

### Edit Annotations

In [ ]:
# Setup Segment Editor
slicer.util.selectModule("SegmentEditor")
segmentEditorWidget = slicer.modules.segmenteditor.widgetRepresentation().self().editor
segmentEditorNode = segmentEditorWidget.mrmlSegmentEditorNode()
segmentEditorWidget.setSegmentationNode(segmentation_node)
segmentEditorWidget.setSourceVolumeNode(image_node)

segmentation = segmentation_node.GetSegmentation()
displayNode = segmentation_node.GetDisplayNode()

# Define the target segment
targetSegmentName = "14"  # Right-Cerebral-White-Matter, see apps/monaibundle/model/wholeBrainSeg_Large_UNEST_segmentation/docs/README.md
targetSegmentID = segmentation.GetSegmentIdBySegmentName(targetSegmentName)

# Set segment visibility (making ONLY the target segment visible)
displayNode.SetAllSegmentsVisibility(False)
displayNode.SetSegmentVisibility(targetSegmentID, True)

# Set the Active Segment for Editing
segmentEditorNode.SetSelectedSegmentID(targetSegmentID)

# Configure viewer
app = slicernb.AppWindow(contents="viewers", windowScale=0.3)
slicernb.setViewLayout(
    "OneUpRedSlice"
)  # other views: OneUpYellowSlice, OneUpGreenSlice, OneUp3D
app.setWindowSize(scale=0.5)
app.setContents("viewers")

# Create an interactive widget for the Red slice view ("R")
liveRedSliceSeg = slicernb.ViewInteractiveWidget(
    "R"
)  # other interactive widgets: Y, G, 1 (3D view)
liveRedSliceSeg.trackMouseMove = False
display(liveRedSliceSeg)

# Add buttons to switch between multiple effects
cutButton = Button(description="Cut")
cutButton.on_click(
    lambda button: segmentEditorWidget.setActiveEffectByName("Scissors")
)  # 2D, 3D view
paintButton = Button(description="Paint")
paintButton.on_click(
    lambda button: segmentEditorWidget.setActiveEffectByName("Paint")
)  # 2D view
eraseButton = Button(description="Erase")
eraseButton.on_click(
    lambda button: segmentEditorWidget.setActiveEffectByName("Erase")
)  # 2D view
HBox([cutButton, paintButton, eraseButton])

#### Edit Annotations in 3D view

In [ ]:
slicernb.setViewLayout("OneUp3D")

# Create an interactive widget for the 3D view ("1")
live3dSeg = slicernb.ViewInteractiveWidget("1")
live3dSeg.trackMouseMove = False
display(live3dSeg)

# Add buttons to switch between multiple effects
cutButton = Button(description="Cut")
cutButton.on_click(lambda button: segmentEditorWidget.setActiveEffectByName("Scissors"))
rotateButton = Button(description="Rotate")
rotateButton.on_click(lambda button: segmentEditorWidget.setActiveEffectByName(""))
HBox([cutButton, rotateButton])

### Save/submit edited label

In [ ]:
labelmapVolumeNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode")
slicer.modules.segmentations.logic().ExportVisibleSegmentsToLabelmapNode(
    segmentation_node, labelmapVolumeNode, image_node
)
slicer.util.saveNode(labelmapVolumeNode, label_file)

ml.save_label(image_id, label_file, params=params)

## 5. Active Learning
Train models anytime when new annotated labels are saved. MONAI Label server will fetch saved final ground truth label and fine-tune the prior model.

For each training loop, the new best metric model will be saved in `<BUNDLE_NAME>/models/model.pt`, in this use case, `apps/monaibundle/model/wholeBrainSeg_Large_UNEST_segmentation/models/model.pt` is saved.

### Run Train Task

In [ ]:
ml.train_start(
    model="wholeBrainSeg_Large_UNEST_segmentation",
    params={"max_epochs": 20, "run_id": "training"},
)

### Check Train Task Status

In [ ]:
ml.train_status(check_if_running=False)

### Stop any running Train Task(s)

In [ ]:
ml.train_stop()

## 6. Repeat the Interactive Labeling Process
Repeat the above process of fetching data and active learning until all unlabeled data are annotated and trained.


## 7. Upload Data to the Project
For any data generated by your notebook that needs to be preserved, upload it to the project before the session ends and the JupyterLab worker terminates. 

In [ ]:
# Upload the labels
! dx upload -r eg_brain_t1_nifti_2/labels

In [ ]:
# Upload updated model
! dx upload apps/monaibundle/model/wholeBrainSeg_Large_UNEST_segmentation/models/model.pt